In [1]:
# 1. Install dependencies and clone FFLUNet
!pip install -q kagglehub nnunetv2 SimpleITK nibabel
!git clone -b add-deep-supervision https://github.com/RAJWARDHAN-B/FFLUNet.git
%cd FFLUNet
!pip install -e .

# Notice the -b flag specifying the branch name!
# !git clone -b add-deep-supervision https://github.com/RAJWARDHAN-B/FFLUNet.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.4 MB/s eta 0:00

In [2]:
# 2. Download BraTS Dataset
import kagglehub
path = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'brats20-dataset-training-validation' dataset.
Dataset downloaded to: /kaggle/input/brats20-dataset-training-validation


In [3]:
# 3. Setup nnUNet workspace
import os
workspace = "/content/nnunet_workspace"
os.environ["nnUNet_raw"] = os.path.join(workspace, "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = os.path.join(workspace, "nnUNet_preprocessed")
os.environ["nnUNet_results"] = os.path.join(workspace, "nnUNet_results")

for folder in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    os.makedirs(os.path.join(workspace, folder), exist_ok=True)

print("nnUNet workspace directories created.")

nnUNet workspace directories created.


In [4]:
# 4. Process Dataset (Label Remap 4->3)
import glob
import shutil
import nibabel as nib
import numpy as np
import json
from tqdm import tqdm

brats_path = os.path.join(path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
patients = sorted(glob.glob(os.path.join(brats_path, "BraTS20_*")))
patients = [p for p in patients if os.path.exists(os.path.join(p, f"{os.path.basename(p)}_seg.nii"))]
print(f"Total valid patients found: {len(patients)}")

dataset_name = "Dataset001_BraTS"
base = os.path.join(os.environ["nnUNet_raw"], dataset_name)
imagesTr = os.path.join(base, "imagesTr")
labelsTr = os.path.join(base, "labelsTr")

shutil.rmtree(base, ignore_errors=True)
os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

valid_cases = 0
for p in tqdm(patients):
    patient_id = os.path.basename(p)
    flair = os.path.join(p, f"{patient_id}_flair.nii")
    t1 = os.path.join(p, f"{patient_id}_t1.nii")
    t1ce = os.path.join(p, f"{patient_id}_t1ce.nii")
    t2 = os.path.join(p, f"{patient_id}_t2.nii")
    seg = os.path.join(p, f"{patient_id}_seg.nii")

    files = [flair, t1, t1ce, t2, seg]
    if not all(os.path.exists(f) for f in files):
        continue

    case_id = f"BraTS_{valid_cases:03d}"
    shutil.copy(flair, os.path.join(imagesTr, f"{case_id}_0000.nii"))
    shutil.copy(t1, os.path.join(imagesTr, f"{case_id}_0001.nii"))
    shutil.copy(t1ce, os.path.join(imagesTr, f"{case_id}_0002.nii"))
    shutil.copy(t2, os.path.join(imagesTr, f"{case_id}_0003.nii"))

    img = nib.load(seg)
    mask = img.get_fdata()
    mask = np.where(mask == 4, 3, mask).astype(np.uint8)
    corrected = nib.Nifti1Image(mask, img.affine, img.header)
    nib.save(corrected, os.path.join(labelsTr, f"{case_id}.nii"))
    valid_cases += 1

dataset_json = {
    "name": "BraTS",
    "channel_names": {"0": "FLAIR", "1": "T1", "2": "T1ce", "3": "T2"},
    "labels": {"background": 0, "necrotic": 1, "edema": 2, "enhancing": 3},
    "numTraining": valid_cases,
    "file_ending": ".nii",
    "overwrite_image_reader_writer": "NibabelIO"
}
with open(os.path.join(base, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=4)
print("dataset.json created with NibabelIO overwrite.")

Total valid patients found: 368


100%|██████████| 368/368 [06:29<00:00,  1.06s/it]

dataset.json created with NibabelIO overwrite.


In [5]:
# 5. Fix PyTorch Version Issue in PolyLR
from pathlib import Path
polylr_path = Path("/content/FFLUNet/nnunetv2/training/lr_scheduler/polylr.py")
if polylr_path.exists():
    text = polylr_path.read_text()
    old = """super().__init__(
            optimizer, current_step if current_step is not None else -1, False
        )"""
    new = """super().__init__(
            optimizer,
            last_epoch=current_step if current_step is not None else -1,
        )"""
    text = text.replace(old, new)
    polylr_path.write_text(text)
    print("Patched polylr.py successfully!")


Patched polylr.py successfully!


In [6]:
# 6. Plan and Preprocess
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_BraTS
Using <class 'nnunetv2.imageio.nibabel_reader_writer.NibabelIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.nibabel_reader_writer.NibabelIO'> reader/writer
100% 368/368 [06:16<00:00,  1.02s/it]
Experiment planning...
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': array([192, 160]), 'median_image_size_in_voxels': array([170., 138.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'UNet_class_name': 'PlainConvUNet', 'UNet_base_num_features': 32, 'n_conv_per_stage_encoder': (2, 2, 2, 2, 2, 2), 'n_conv_per_stage_decoder': (2, 2, 2, 2, 2), 'num_pool_per_axis': [5, 

In [ ]:
# 7. Train Model (FFLUNet with Deep Supervision)
!nnUNetv2_train 1 3d_fullres 0 -tr nnUNetTrainer_FFLUNetDS

2026-06-30 18:07:20.424782: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device: cuda:0
/content/FFLUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler() if self.device.type == "cuda" else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#########################################################